# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrathibhaShaliniS/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

A page is worth reviewing first if it's getting real search visibility, and it's either stale (not updated in a long time) or under-clicking for how well it ranks. Pages that are both rank highest.

In [2]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/PrathibhaShaliniS/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["ctr_gap"] = (
    (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.50)
).astype(int)

def reason_code(row):
    if row["stale"] and row["ctr_gap"]:
        return "stale_and_ctr_gap"
    if row["stale"]:
        return "stale_visible_page"
    if row["ctr_gap"]:
        return "ctr_underperforming_position"
    return "not_flagged"

df["reason_code"] = df.apply(reason_code, axis=1)
print(df["reason_code"].value_counts())

reason_code
ctr_underperforming_position    16463
not_flagged                     13363
stale_and_ctr_gap                  95
stale_visible_page                 79
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score every page, attach the reason code and action label, rank everything, and write the ranked queue to work/outputs/baseline_action_score.csv. No future-window or label-derived columns go into the score — only days_since_last_update, impressions_90d, avg_position, and ctr, all real and known before any decision point.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
ACTION_BY_REASON = {
    "stale_and_ctr_gap": "refresh_and_fix_ctr",
    "stale_visible_page": "refresh",
    "ctr_underperforming_position": "review_snippet_ctr",
    "not_flagged": "monitor",
}

df["baseline_score"] = df["visible"] * df["impressions_90d"] * (df["stale"] + df["ctr_gap"])
df["action"] = df["reason_code"].map(ACTION_BY_REASON)
df["rank"] = df["baseline_score"].rank(method="first", ascending=False).astype(int)

out_cols = [
    "content_id", "client_id", "rank", "baseline_score", "reason_code", "action",
    "impressions_90d", "clicks_90d", "ctr", "avg_position", "days_since_last_update",
    "content_age_days",
]
out = df[out_cols].sort_values("rank")

os.makedirs("work/outputs", exist_ok=True)
out.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(out)} rows to work/outputs/baseline_action_score.csv")
out.head(10)

Wrote 30000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,rank,baseline_score,reason_code,action,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,content_age_days
6653,content_5fe46e04994d,client_4e07408562,1,517715,ctr_underperforming_position,review_snippet_ctr,517715,741,0.14,4.2,104,537
17812,content_aaef01a50def,client_19581e27de,2,517109,ctr_underperforming_position,review_snippet_ctr,517109,1270,0.25,5.4,22,445
26844,content_8c19996aa890,client_4e07408562,3,509252,ctr_underperforming_position,review_snippet_ctr,509252,785,0.15,2.5,20,445
21819,content_4c36c775b818,client_4e07408562,4,463103,ctr_underperforming_position,review_snippet_ctr,463103,1889,0.41,2.3,20,445
29879,content_1a9e894be2e2,client_19581e27de,5,416180,ctr_underperforming_position,review_snippet_ctr,416180,944,0.23,4.0,22,482
18870,content_db5989a78dd3,client_4e07408562,6,345111,ctr_underperforming_position,review_snippet_ctr,345111,733,0.21,5.4,20,445
26531,content_cb112fce36be,client_19581e27de,7,309910,ctr_underperforming_position,review_snippet_ctr,309910,492,0.16,5.6,104,126
3394,content_36ff89c8214e,client_19581e27de,8,295097,ctr_underperforming_position,review_snippet_ctr,295097,154,0.05,7.3,104,144
7678,content_8451fc6f034d,client_d029fa3a95,9,272144,ctr_underperforming_position,review_snippet_ctr,272144,75,0.03,2.3,20,280
27478,content_008fb02c46cb,client_349c41201b,10,236803,ctr_underperforming_position,review_snippet_ctr,236803,605,0.26,4.4,20,111


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Leakage check: the score only uses days_since_last_update, impressions_90d, avg_position, and ctr — all real, observable signals, none of them future-window columns or FlyRank's own product flags (trend_direction, trend_pct, is_declining_label are never used as inputs).
**Weak picks:**
The score multiplies straight through by raw impressions_90d (heavy-tailed), so the top of the ranking is dominated by a handful of huge pages that all share the same reason code — smaller stale_and_ctr_gap pages never make it near the top even though they may be a cleaner fix.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
score_inputs = {"days_since_last_update", "impressions_90d", "avg_position", "ctr"}
banned = {"trend_direction", "trend_pct", "is_declining_label"}
assert banned.isdisjoint(score_inputs), "leakage: a label-derived column is in the score inputs"
print("No future-window or product-flag columns used in the score. Inputs:", score_inputs)


No future-window or product-flag columns used in the score. Inputs: {'ctr', 'days_since_last_update', 'impressions_90d', 'avg_position'}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.